In [4]:
import pandas as pd
import numpy as np
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [5]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kithm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kithm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kithm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kithm\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kithm\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [6]:
df = pd.read_csv("../data/sms_spam_cleaned.csv")

In [7]:
print(df.columns)

Index(['label', 'message', 'character_count', 'word_count', 'sentence_count'], dtype='str')


In [8]:
df.head()

,label,message,character_count,word_count,sentence_count
0,ham,"Go until jurong point, crazy.. Available only ...",111,24,2
1,ham,Ok lar... Joking wif u oni...,29,8,2
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,37,2
3,ham,U dun say so early hor... U c already then say...,49,13,1
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,15,1


In [9]:
df["original_message"] = df["message"]

In [10]:
df[["label", "original_message"]].head()

,label,original_message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Understand preprocessing
First Lowercase conversion

In [11]:
def preprocess_text(text):

    text = text.lower()

    return text

In [12]:
sample = "Congratulations! You Won $1000."

print(preprocess_text(sample))

congratulations! you won $1000.


In [13]:
# Initialize preprocessing tools
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """
    Clean and preprocess an SMS message.

    Steps:
    1. Convert to lowercase
    2. Replace URLs
    3. Replace email addresses
    4. Replace money symbols
    5. Remove numbers, punctuation and special characters
    6. Tokenize
    7. Remove stopwords
    8. Lemmatize
    9. Join tokens back into a string
    """

    # Handle missing or invalid values
    if pd.isna(text):
        return ""

    # Make sure the input is a string
    text = str(text)

    # Convert text to lowercase
    text = text.lower()

    # Replace URLs with a meaningful token
    text = re.sub(r"http\S+|www\S+", " url ", text)

    # Replace email addresses
    text = re.sub(r"\S+@\S+\.\S+", " email ", text)

    # Replace money symbols
    text = re.sub(r"[$£€₹]", " money ", text)

    # Remove numbers, punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize the text
    tokens = word_tokenize(text)

    # Remove stopwords and very short tokens
    tokens = [
        word
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    # Lemmatize each token
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    # Join the words back into one cleaned string
    cleaned_text = " ".join(tokens)

    return cleaned_text

In [14]:
sample_message = """
Congratulations!!! You have won £500.
Visit https://example.com and claim your prize now!
"""

cleaned_sample = preprocess_text(sample_message)

print("Original message:")
print(sample_message)

print("\nCleaned message:")
print(cleaned_sample)

Original message:

Congratulations!!! You have won £500.
Visit https://example.com and claim your prize now!


Cleaned message:
congratulation money visit url claim prize


In [15]:
df["clean_message"] = df["message"].apply(preprocess_text)

In [16]:
df[
    ["label", "original_message", "clean_message"]
].head(10)

,label,original_message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis great wo...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,dun say early hor already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though
5,spam,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey darling week word back like fun st...
6,ham,Even my brother is not like to speak with me. ...,even brother like speak treat like aid patent
7,ham,As per your request 'Melle Melle (Oru Minnamin...,per request melle melle oru minnaminunginte nu...
8,spam,WINNER!! As a valued network customer you have...,winner valued network customer selected receiv...
9,spam,Had your mobile 11 months or more? U R entitle...,mobile month entitled update latest colour mob...


In [17]:
df[
    ["label", "original_message", "clean_message"]
].sample(10, random_state=42)

,label,original_message,clean_message
1566,ham,"K, makes sense, btw carlos is being difficult ...",make sense btw carlos difficult guy gon na smo...
1988,spam,"URGENT! Your mobile No *********** WON a £2,00...",urgent mobile money bonus caller prize nd atte...
1235,ham,If you still havent collected the dough pls le...,still havent collected dough pls let know go p...
2868,ham,Wat time do u wan 2 meet me later?,wat time wan meet later
3435,spam,"You can stop further club tones by replying ""S...",stop club tone replying stop mix see tone com ...
1471,ham,Check wid corect speling i.e. Sarcasm,check wid corect speling sarcasm
1129,ham,Hey! There's veggie pizza... :/,hey veggie pizza
3747,ham,"Hey, I missed you tm of last night as my phone...",hey missed tm last night phone charge smile me...
3047,spam,URGENT! We are trying to contact U. Todays dra...,urgent trying contact today draw show money pr...
530,ham,Ummmmmaah Many many happy returns of d day my ...,ummmmmaah many many happy return day dear swee...


In [18]:
empty_cleaned_messages = df[
    df["clean_message"].str.strip() == ""
]

print(
    "Number of empty cleaned messages:",
    len(empty_cleaned_messages)
)

Number of empty cleaned messages: 12


In [19]:
empty_cleaned_messages[
    ["label", "original_message", "clean_message"]
].head(20)

,label,original_message,clean_message
249,ham,What you doing?how are you?,
788,ham,K I'll be there before 4.,
940,ham,Where @,
1504,ham,U too...,
1561,ham,645,
2677,ham,Can a not?,
3193,ham,:),
4021,ham,G.W.R,
4278,ham,:( but your not here....,
4502,ham,:-) :-),


In [20]:
df = df[
    df["clean_message"].str.strip() != ""
].copy()

In [21]:
df.reset_index(drop=True, inplace=True)

In [22]:
print("Dataset shape after removing empty messages:", df.shape)

Dataset shape after removing empty messages: (5157, 7)


In [23]:
df[
    ["label", "original_message", "clean_message"]
].isnull().sum()

label               0
original_message    0
clean_message       0
dtype: int64

In [24]:
clean_duplicates = df.duplicated(
    subset=["clean_message"]
).sum()

print(
    "Duplicate cleaned messages:",
    clean_duplicates
)

Duplicate cleaned messages: 113


In [25]:
df[
    df.duplicated(
        subset=["clean_message"],
        keep=False
    )
][
    ["label", "original_message", "clean_message"]
].sort_values("clean_message").head(20)

,label,original_message,clean_message
805,spam,25p 4 alfie Moon's Children in need song on ur...,alfie moon child need song ur mob tell ur txt ...
4716,spam,5p 4 alfie Moon's Children in need song on ur ...,alfie moon child need song ur mob tell ur txt ...
3409,spam,8007 25p 4 Alfie Moon's Children in Need song ...,alfie moon child need song ur mob tell ur txt ...
4181,ham,I anything lor.,anything lor
3384,ham,I anything lor...,anything lor
74,ham,U can call me now...,call
4004,ham,U can call now...,call
3541,ham,Where are you call me.,call
2052,ham,Then why you came to hostel.,came hostel
1551,ham,When you came to hostel.,came hostel


In [26]:
df = df.drop_duplicates(
    subset=["label", "clean_message"]
).copy()

In [27]:
df.reset_index(drop=True, inplace=True)

In [28]:
df["clean_char_length"] = df["clean_message"].str.len()

df["clean_word_count"] = df["clean_message"].apply(
    lambda text: len(text.split())
)

In [29]:
df[
    [
        "label",
        "original_message",
        "clean_message",
        "clean_char_length",
        "clean_word_count"
    ]
].head()

,label,original_message,clean_message,clean_char_length,clean_word_count
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis great wo...,78,14
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif oni,21,5
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...,99,20
3,ham,U dun say so early hor... U c already then say...,dun say early hor already say,29,6
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though,35,7


In [30]:
df["clean_message"]

0       go jurong point crazy available bugis great wo...
1                                   ok lar joking wif oni
2       free entry wkly comp win fa cup final tkts st ...
3                           dun say early hor already say
4                     nah think go usf life around though
                              ...                        
5039    nd time tried contact money pound prize claim ...
5040                              going esplanade fr home
5041                                 pity mood suggestion
5042    guy bitching acted like interested buying some...
5043                                       rofl true name
Name: clean_message, Length: 5044, dtype: str

In [31]:
def preprocess_text_for_gru(text):
    """
    Perform light text cleaning for the GRU model.

    Stopwords are retained because word order and sentence
    structure can be useful for sequence models.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Replace URLs and email addresses
    text = re.sub(r"http\S+|www\S+", " url ", text)
    text = re.sub(r"\S+@\S+\.\S+", " email ", text)

    # Replace money symbols
    text = re.sub(r"[$£€₹]", " money ", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [32]:
df["gru_message"] = df["original_message"].apply(
    preprocess_text_for_gru
)

In [33]:
df[
    [
        "original_message",
        "clean_message",
        "gru_message"
    ]
].sample(10, random_state=42)

,original_message,clean_message,gru_message
3654,Thanks for your message. I really appreciate y...,thanks message really appreciate sacrifice sur...,thanks for your message i really appreciate yo...
4521,"What i mean was i left too early to check, cos...",mean left early check co working,what i mean was i left too early to check cos ...
1049,Send to someone else :-),send someone else,send to someone else
3179,Then what about further plan?,plan,then what about further plan
3313,I sent your maga that money yesterday oh.,sent maga money yesterday oh,i sent your maga that money yesterday oh
3422,Aight do you still want to get money,aight still want get money,aight do you still want to get money
240,Although i told u dat i'm into baig face watch...,although told dat baig face watch really like ...,although i told u dat i m into baig face watch...
2052,I'm not coming home 4 dinner.,coming home dinner,i m not coming home dinner
4481,We took hooch for a walk toaday and i fell ove...,took hooch walk toaday fell splat grazed knee ...,we took hooch for a walk toaday and i fell ove...
239,"Okay. No no, just shining on. That was meant t...",okay shining meant signing sound better,okay no no just shining on that was meant to b...


In [34]:
df["label_encoded"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

In [35]:
df[
    ["label", "label_encoded"]
].drop_duplicates()

,label,label_encoded
0,ham,0
2,spam,1


In [36]:
print(
    "Missing encoded labels:",
    df["label_encoded"].isnull().sum()
)

Missing encoded labels: 0


In [37]:
print("Final dataset shape:", df.shape)

print("\nFinal columns:")
print(df.columns.tolist())

df.head()

Final dataset shape: (5044, 11)

Final columns:
['label', 'message', 'character_count', 'word_count', 'sentence_count', 'original_message', 'clean_message', 'clean_char_length', 'clean_word_count', 'gru_message', 'label_encoded']


,label,message,character_count,word_count,sentence_count,original_message,clean_message,clean_char_length,clean_word_count,gru_message,label_encoded
0,ham,"Go until jurong point, crazy.. Available only ...",111,24,2,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis great wo...,78,14,go until jurong point crazy available only in ...,0
1,ham,Ok lar... Joking wif u oni...,29,8,2,Ok lar... Joking wif u oni...,ok lar joking wif oni,21,5,ok lar joking wif u oni,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,37,2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...,99,20,free entry in a wkly comp to win fa cup final ...,1
3,ham,U dun say so early hor... U c already then say...,49,13,1,U dun say so early hor... U c already then say...,dun say early hor already say,29,6,u dun say so early hor u c already then say,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,15,1,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though,35,7,nah i don t think he goes to usf he lives arou...,0


In [38]:
df.to_csv(
    "../data/preprocessed_sms_spam.csv",
    index=False
)

print("Preprocessed dataset saved successfully.")

Preprocessed dataset saved successfully.


In [39]:
import os

file_path = "../data/preprocessed_sms_spam.csv"

print("File exists:", os.path.exists(file_path))

File exists: True


In [40]:
print("PREPROCESSING SUMMARY")
print("=" * 50)

print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

print("\nMissing values:")
print(
    df[
        [
            "label",
            "original_message",
            "clean_message",
            "gru_message",
            "label_encoded"
        ]
    ].isnull().sum()
)

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nEncoded class distribution:")
print(df["label_encoded"].value_counts())

print(
    "\nEmpty SVM messages:",
    (df["clean_message"].str.strip() == "").sum()
)

print(
    "Empty GRU messages:",
    (df["gru_message"].str.strip() == "").sum()
)

print(
    "\nDuplicate label cleaned message pairs:",
    df.duplicated(
        subset=["label", "clean_message"]
    ).sum()
)

PREPROCESSING SUMMARY
Number of records: 5044
Number of columns: 11

Missing values:
label               0
original_message    0
clean_message       0
gru_message         0
label_encoded       0
dtype: int64

Class distribution:
label
ham     4465
spam     579
Name: count, dtype: int64

Encoded class distribution:
label_encoded
0    4465
1     579
Name: count, dtype: int64

Empty SVM messages: 0
Empty GRU messages: 0

Duplicate label cleaned message pairs: 0
